# EDA básico — HumanEval y MBPP

**Antes de ejecutar:** desde la raíz del proyecto,

`pip install -r requirements.txt`

`python src/ingest_datasets_v0.py`

Así existen `data/raw/humaneval/`, `data/raw/mbpp/` y el manifiesto.

In [ ]:
from pathlib import Path
import json

import pandas as pd

# Raíz del repositorio (carpeta padre de notebooks/)
ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
MANIFEST = RAW / "manifest_v0.json"

print("ROOT:", ROOT)
print("Existe manifiesto:", MANIFEST.is_file())

In [ ]:
if MANIFEST.is_file():
    with open(MANIFEST, encoding="utf-8") as f:
        manifest = json.load(f)
    print("Versión ingesta:", manifest.get("ingestion_version"))
    print("Inicio (UTC):", manifest.get("execution_started_utc"))
    print("Conteos:", manifest.get("row_counts"))
else:
    print("Falta manifest_v0.json — ejecuta la ingesta primero.")

In [ ]:
def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

he_path = RAW / "humaneval" / "openai_humaneval_test.jsonl"
if he_path.is_file():
    df_he = load_jsonl(he_path)
    display(df_he.head(2))
    print("Filas HumanEval:", len(df_he))
    print("Columnas:", list(df_he.columns))
    if "prompt" in df_he.columns:
        df_he["n_chars_prompt"] = df_he["prompt"].astype(str).str.len()
        display(df_he["n_chars_prompt"].describe())
else:
    print(f"No encontrado: {he_path} — corre la ingesta.")

In [ ]:
mb_path = RAW / "mbpp" / "mbpp.jsonl"
if mb_path.is_file():
    df_mb = load_jsonl(mb_path)
    display(df_mb.head(2))
    print("Filas MBPP:", len(df_mb))
    print("Columnas:", list(df_mb.columns))
    if "text" in df_mb.columns:
        df_mb["n_chars_enunciado"] = df_mb["text"].astype(str).str.len()
        display(df_mb["n_chars_enunciado"].describe())
else:
    print(f"No encontrado: {mb_path} — corre la ingesta.")